# Lesson 13 · Skip connections & ResNet: one addition that lets deep networks learn

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/13-resnet-en.ipynb)

This notebook accompanies [Lesson 13 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/13-resnet.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
Lesson 12 ended with a paradox: on MNIST, a 56-layer “plain” convolutional network has a higher *training* error than
a 20-layer one. We write the skip connection $h \leftarrow h + F(h)$ ourselves in numpy, put it into exactly the deep
network of Lessons 09–10, then see how the gradient, the signal and the training results change when the only change
is a single addition.

**What you'll do:**
1. Revisit the paradox of deep plain networks, and why it is not overfitting.
2. Write a residual block, and check that the derivative of $x + F(x)$ has an identity part.
3. Measure the gradient of each layer of a 30-layer MLP at initialization, with and without skip connections, on
   exactly the demo's 64 images.
4. Train two 30-layer networks for real, for 1 epoch, then read the results of the full sweep (MLPs up to 100 layers,
   CNNs up to 56 layers).
5. Delete the blocks of a trained ResNet one at a time (the experiment of Veit et al., 2016) and compare with a plain
   network.

**Background:** [Lesson 09 · Vanishing gradients & initialization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/),
[Lesson 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chuan-hoa/) and
[Lesson 12 · Pooling & classic CNN architectures](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/kien-truc-cnn/).
**Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

Every number in the notebook is computed by the cells below (or read from the results files the demo uses), and
matches the numbers in the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign. The function `sig()` prints the
gradient ratios exactly as the demo shows them (899 · 38.6 · 7.3·10⁸).

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import math
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding:
# with skip connections = blue, solid line, round dots; plain network = orange, dashed line, squares (as in the demo)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
RES = {"color": BLUE, "ls": "-", "marker": "o"}
PLAIN = {"color": ORANGE, "ls": (0, (5, 3)), "marker": "s"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"  # never prints “−0”
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def num_eta(e):
    """η with one significant digit: 0.3 · 0.03 · 0.01."""
    return num(e, max(0, -math.floor(math.log10(e) + 1e-9)))


def sig(x):
    """As in the demo: 0.451 · 5.6 · 38.6 · 899 · 8,720; from 10,000 up, the power-of-10 form: 7.3·10⁸."""
    a = abs(x)
    if a == 0:
        return "0"
    if 1e-3 <= a < 1e4:
        return num(x, 4 if a < 0.01 else 3 if a < 1 else 1 if a < 100 else 0)
    e = math.floor(math.log10(a))
    return f"{num(x / 10**e, 1)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 0.001 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def pow_tick(v, _pos=None):
    return "1" if abs(math.log10(v)) < 1e-9 else f"10{str(round(math.log10(v))).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis in the English format. Log axes: 1–2–5 ticks if short, powers of 10 if long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                long = hi / lo >= 1000
                step = max(1, math.ceil(math.log10(hi / lo) / 6)) if long else 1
                ticks = [m * 10.0**k for k in range(-15, 16, step) for m in ((1,) if long else (1, 2, 5))]
                axis.set_major_locator(FixedLocator([t for t in ticks if lo <= t <= hi]))
                axis.set_minor_formatter(NullFormatter())
                axis.set_major_formatter(FuncFormatter(pow_tick if long else tick))
            elif isinstance(axis.get_major_formatter(), ScalarFormatter):  # text labels stay (set_xticks(..., labels))
                axis.set_major_formatter(FuncFormatter(tick))


print(num(13007, 0), num(0.1, 1), pct(0.9608), sig(899.3), sig(38.56), sig(7.348e8), sig(0.4512))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. As in Lessons 09–10, we split them:
- **training:** the first 55,000 images of the training file;
- **validation:** the last 5,000 images, to *choose* the learning rate $\eta$ and to report the accuracy of the
  1-epoch runs.

The cell also loads three results files that the demo uses (`fetch_json`, with an MD5 check). They are sweeps that
took from a few minutes to more than half an hour: deep MLPs of 10–100 layers (`mnist-resmlp.json`) and convolutional
networks of 8–56 layers, both plain (`mnist-depth.json`, Lesson 12) and with skip connections (`mnist-resnet.json`).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of a long training run (the same file as the demo on the web). build.py replaces COURSE_DATA_URL with the
# data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
mlp_runs = fetch_json("mnist-resmlp.json", "b05cf3f5374b56df6f33f3529031e117")  # deep MLPs: plain / skip connections
cnn_plain = fetch_json("mnist-depth.json", "8156284d0611a3445408647bbae5d813")  # Lesson 12's plain CNN
cnn_res = fetch_json("mnist-resnet.json", "c52fd453cf49dd42e7b94e6e6486fdb0")  # the same CNN, plus skip connections

In [ ]:
train_img, train_lab = load("train")
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
N_TRAIN, BATCH = 55000, 64
Xtr, ytr = X_all[:N_TRAIN], y_all[:N_TRAIN]  # training
Xv, yv = X_all[N_TRAIN:], y_all[N_TRAIN:]  # validation
print(f"training {num(len(ytr), 0)} · validation {num(len(yv), 0)} images")
assert (len(ytr), len(yv), BATCH) == (mlp_runs["nTrain"], mlp_runs["nVal"], mlp_runs["batch"])

The demo's gradient measurements use **the first training batch of seed 1**: the first 64 images in the shuffled
order `default_rng(1).permutation(55000)`. The cell below picks exactly that batch again from MNIST and compares it
with the 64 images the demo stores.

In [ ]:
idx_b = np.random.default_rng(1).permutation(N_TRAIN)[:BATCH]
xb, yb = Xtr[idx_b], ytr[idx_b]
demo_imgs = np.frombuffer(base64.b64decode(mlp_runs["images"]), np.uint8).reshape(BATCH, 784)
assert np.array_equal(demo_imgs, train_img.reshape(-1, 784)[idx_b]) and yb.tolist() == mlp_runs["labels"]
print("exactly the demo's 64 images; labels:", " ".join(map(str, yb[:16])), "…")

fig, axes = plt.subplots(4, 16, figsize=(10, 2.9))
for ax, x, lab in zip(axes.flat, xb, yb):
    ax.imshow(x.reshape(28, 28), cmap="gray_r")
    ax.set_title(str(lab), fontsize=9, pad=2)
    ax.axis("off")
fig.suptitle(
    "The batch of 64 images used to measure the gradient at initialization (the first batch of seed 1)", fontsize=11
)
plt.tight_layout()
fmt_axes(fig)
plt.show()

## 3 · The paradox: more layers, a higher training error

Lesson 12 built the “wall of depth” following exactly the recipe of He et al. (2015) for CIFAR-10, scaled down for
MNIST: a first 3×3 convolutional layer, three stages of 3×3 convolutions (8, 16, 32 channels), each layer being
Conv → BatchNorm → ReLU, then an average over the whole image and a dense layer with 10 outputs. Depths of 8, 20, 32,
44, 56 layers; SGD, batch 64, 3 epochs, 3 seeds; $\eta$ chosen for each depth on the validation set. The cell below
reads those results back and chooses $\eta$ exactly as the demo does: the $\eta$ with the lowest median validation
error.

The number to look at is the **training error** (measured after training, on the first 10,000 training images), not
the test error.

In [ ]:
def best_cnn(grid):
    """For each depth: the η with the lowest median validation error (a seed that blew up counts as 100%), as in the
    demo."""
    out = {}
    for d in grid["depths"]:
        cfg = {r["eta"]: r["seeds"] for r in grid["runs"] if r["depth"] == d}
        eta = min(grid["etas"], key=lambda e: np.median([s.get("val", 1.0) for s in cfg[e]]))
        out[d] = {"eta": eta, **{k: [s[k] for s in cfg[eta] if k in s] for k in ("train", "val", "test", "params")}}
    return out


def seed_text(v, d=2):
    """The median [min–max] of the seeds."""
    return f"{pct(np.median(v), d)} [{pct(min(v), d)}–{pct(max(v), d)}]"


P_cnn = best_cnn(cnn_plain)
print(f"training error measured on {num(cnn_plain['nTrainEval'], 0)} training images, {cnn_plain['epochs']} epochs")
for d, r in P_cnn.items():
    print(
        f"plain CNN, {d:2d} layers ({num(r['params'][0], 0):>7} parameters), η = {num_eta(r['eta']):<4}: training error "
        f"{seed_text(r['train'])}, test error {seed_text(r['test'])}"
    )
long_plain = {(q["depth"], q["eta"]): q for q in cnn_plain["long"]}
for d in (20, 56):
    q = min((q for (dd, _), q in long_plain.items() if dd == d), key=lambda q: q["val"])
    print(f"10 epochs, seed 0, {d} layers (η = {num_eta(q['eta'])}): training error {pct(q['train'])}")

The figure below plots the training error against depth: each orange square is one seed, and the dashed line connects
the medians.

In [ ]:
depths = cnn_plain["depths"]
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for i, d in enumerate(depths):
    ax.scatter(
        [d] * len(P_cnn[d]["train"]), 100 * np.array(P_cnn[d]["train"]), color=ORANGE, marker="s", s=28, zorder=3
    )
ax.plot(depths, [100 * np.median(P_cnn[d]["train"]) for d in depths], color=ORANGE, ls=PLAIN["ls"], lw=1.6)
ax.set(xticks=depths, xlabel="number of layers", ylabel="training error (%)", ylim=(0, None))
ax.set_title("Plain CNN on MNIST (Lesson 12): deeper, a higher training error")
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()

In [ ]:
def best_mlp(arch, norm, depth):
    """The η with the highest median validation accuracy (a seed that blew up counts as 0), as in the demo; returns η
    and the seeds."""
    rs = [r for r in mlp_runs["runs"] if (r["arch"], r["norm"], r["depth"]) == (arch, norm, depth)]
    best = max(rs, key=lambda r: np.median([s.get("val", 0.0) for s in r["seeds"]]))
    return best["eta"], best["seeds"]


def mlp_text(seeds):
    v, n = [s["val"] for s in seeds if "val" in s], [s["nanStep"] for s in seeds if "nanStep" in s]
    if not v:
        return f"blew up (NaN) at steps {min(n)}–{max(n)}" if min(n) != max(n) else f"blew up (NaN) at step {n[0]}"
    return seed_text(v, 1) + (f" ({len(n)}/{len(seeds)} seeds blew up)" if n else "")


d20, d56 = P_cnn[20]["train"], P_cnn[56]["train"]
print(
    f"CNN: 56 layers have a higher training error than 20 layers for all 3 seeds: {'yes' if min(d56) > max(d20) else 'no'}"
)
for norm in ("none", "bn"):
    eta, seeds = best_mlp("plain", norm, 30)
    print(
        f"plain 30-layer MLP, {'BatchNorm' if norm == 'bn' else 'no normalization'} (η = {num_eta(eta)}): "
        f"{mlp_text(seeds)} correct on the validation set after 1 epoch"
    )

A 56-layer network contains the 20-layer network: if the extra layers just compute the identity mapping (output equal
to input), it is exactly as good as the 20-layer network. And yet after training, its *training* error is higher, for
all 3 seeds. This is not **overfitting** (Lesson 08): overfitting means a low training error but a high test error.
Here, the optimization itself fails. He et al. (2015) call this the **degradation problem**, and they see it on
ImageNet too: a 34-layer plain network has a top‑1 error of 28.54%, worse than an 18-layer plain network (27.94%), and
its training error is higher throughout training (Table 2 and Figure 4 of the paper).

Three familiar explanations all fail to fit:
- *“It hasn't trained long enough.”* 10 epochs (the cell above) don't close the gap; the paper also tried training for
  3 times as many steps.
- *“The gradient vanishes.”* All of these networks have BatchNorm; He et al. checked that neither the forward signal
  nor the backward gradient vanishes. Section 5 will show that what happens at initialization is even the opposite:
  the gradient *explodes*.
- *“Adding BatchNorm is enough.”* Lesson 10 saw this with the 30-layer MLP (the last two lines of the cell above):
  with or without BatchNorm, the plain network stays far from the 96% of the 10-layer network (section 8). 66.0%
  versus 77.0% is within the variation across seeds, so we cannot say whether BatchNorm makes it better or worse; we
  only know that it doesn't rescue it.

## 4 · The residual block: $y = x + F(x)$

ResNet's idea: don't make a few layers learn the desired mapping $H(x)$ directly; let them learn **the difference**
$F(x) = H(x) - x$, then add $x$ back through a **skip connection** (shortcut):

$$y = x + F(x)$$

The addition has no parameters at all. We use the “pre-activation” form of He et al. (2016), exactly like the demo's
MLP: the branch $F$ is normalization → ReLU → linear layer, $F(x) = \mathrm{ReLU}(N(x))\,W + b$, and the shortcut is a
pure identity. In this section we leave out $N$ to keep things short. Compared with a “plain” layer
$y = \mathrm{ReLU}(xW + b)$ with the same $W$, $b$:

### 🤔 Predict first
Set every weight and bias to 0: $W = 0$, $b = 0$. What does the plain layer return? What does the residual block
return? And how much of the gradient $\partial L/\partial x$ is left after passing through each one?

In [ ]:
def relu(a):
    return np.maximum(0, a)


def vec(a, d=2):
    return "[" + ", ".join(num(v, d) for v in a) + "]"


x = np.random.default_rng(0).normal(size=(1, 5))  # one example, 5 units
W0, b0 = np.zeros((5, 5)), np.zeros(5)
dy = np.ones((1, 5))  # the gradient ∂L/∂y sent back from the layers after it

y_plain = relu(x @ W0 + b0)
y_res = x + relu(x) @ W0 + b0
dx_plain = (dy * (x @ W0 + b0 > 0)) @ W0.T  # backward pass through ReLU(xW + b)
dx_res = dy + (dy @ W0.T) * (x > 0)  # backward pass through x + ReLU(x)W + b: the shortcut + the branch F
print("x                        =", vec(x[0]))
print("plain layer,    W = 0: y =", vec(y_plain[0]), "  ∂L/∂x =", vec(dx_plain[0]))
print("residual block, W = 0: y =", vec(y_res[0]), "  ∂L/∂x =", vec(dx_res[0]))
assert np.array_equal(y_res, x) and np.array_equal(dx_res, dy) and not y_plain.any() and not dx_plain.any()

With $W = 0$, the plain layer wipes out the signal entirely: $y = \mathrm{ReLU}(0) = 0$, and the gradient it sends
back is 0 too. The residual block returns exactly $x$, and the gradient passes through intact. In other words,
**“doing nothing” is the easiest state** of a residual block: if a block doesn't help, training only has to pull $F$
close to 0. A stack of plain layers that wants to compute the identity mapping has to learn
$\mathrm{ReLU}(xW + b) = x$ on its own, which is much harder.

More generally, with the row-vector convention $y_j = x_j + \sum_i \mathrm{ReLU}(x_i)\,W_{ij} + b_j$, so

$$\frac{\partial y_j}{\partial x_i} = \underbrace{\delta_{ij}}_{\text{shortcut}} + \underbrace{\mathbb 1[x_i > 0]\,W_{ij}}_{\text{branch } F}
\qquad\Longrightarrow\qquad \frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} + \Big(\frac{\partial L}{\partial y}\,W^\top\Big) \odot \mathbb 1[x > 0]$$

The Jacobian matrix is $I + \partial F/\partial x$: the $I$ part carries the gradient back to the previous layer
*without going through any multiplication*. The cell below checks that formula with finite differences, for a random
$W$.

In [ ]:
rng = np.random.default_rng(1)
W1, b1 = rng.normal(0, 0.5, (5, 5)), rng.normal(0, 0.1, 5)


def block(x):
    return x + relu(x) @ W1 + b1


J_formula = np.eye(5) + (x[0] > 0)[:, None] * W1  # J[i, j] = ∂y_j/∂x_i
J_numeric = np.zeros((5, 5))
for i in range(5):
    e = np.zeros((1, 5))
    e[0, i] = 1e-6
    J_numeric[i] = (block(x + e) - block(x - e))[0] / 2e-6
print(f"largest difference between the formula and finite differences: {sig(np.abs(J_formula - J_numeric).max())}")
assert np.allclose(J_formula, J_numeric, atol=1e-8)
print("diagonal of J (always has the 1 from the shortcut):", vec(np.diag(J_formula)))

## 5 · The gradient through the addition: a 30-layer MLP at initialization

Now we put that block into a real network: the deep MLP of Lessons 09–10, 784 → 30×64 → 10, with He initialization.
We build two networks with **exactly the same weights**, differing only in the addition:
- **plain:** $h \leftarrow \mathrm{ReLU}(N(hW + b))$ at every layer;
- **with skip connections:** $h \leftarrow h + \mathrm{ReLU}(N(h))\,W + b$ from layer 2 on (pre-activation blocks);
  layer 1 (784 → 64) is still a plain layer because the size changes, and $N$ → ReLU runs one more time before the
  output layer.

$N$ is BatchNorm (Lesson 10) or nothing at all. So that the networks match the demo number for number, the weights
are drawn the way the demo draws them in the browser: JavaScript's `mulberry32(1)` generator, then the Box–Muller
transform turns each pair of uniform numbers into one normal number.

In [ ]:
WIDTH, EPS, MOM = 64, 1e-5, 0.1  # as in Lesson 10: the width, BatchNorm's ε, the momentum of the running mean


def mulberry32(seed, n):
    """The first n uniform numbers in [0, 1) of mulberry32(seed), the generator the demo uses (rewritten in numpy)."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def he_init(depth, seed=1):
    """The demo's order: ε_out (64×10), ε₁ (784×64), ε₂, ε₃… (64×64), taken one after another from one stream of
    normal numbers. A deeper network only appends layers, so the first 30 layers of the 100-layer network are the
    30-layer network's."""
    shapes = [(WIDTH, 10), (784, WIDTH)] + [(WIDTH, WIDTH)] * (depth - 1)
    sizes = [r * c for r, c in shapes]
    u = mulberry32(seed, 2 * sum(sizes))
    z = np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller
    eps = [z[k - n : k].reshape(s) for s, n, k in zip(shapes, sizes, itertools.accumulate(sizes))]
    return [np.sqrt(2 / r) * e for (r, _), e in zip(shapes[1:], eps[1:])] + [np.sqrt(2 / WIDTH) * eps[0]]  # He


W_demo = he_init(30)
print("W₁:", W_demo[0].shape, " W₂…W₃₀:", W_demo[1].shape, "×", len(W_demo) - 2, " W_out:", W_demo[-1].shape)
print(f"standard deviation of W₁: {num(W_demo[0].std(), 4)} (He: √(2/784) = {num(np.sqrt(2 / 784), 4)})")

The two networks share one list of weights; BatchNorm starts with $\gamma = 1$, $\beta = 0$ and keeps a running mean
(momentum 0.1) for use at inference time, as in Lesson 10. The network with skip connections needs $L + 1$
normalization layers: layer 1, one before the branch of each block, and the last one.

In [ ]:
def make_net(arch, norm, depth, seed=1):
    k = depth + (arch == "res")  # the number of normalization layers
    W = he_init(depth, seed)
    return {
        "arch": arch,
        "norm": norm,
        "depth": depth,
        "W": W,
        "b": [np.zeros(w.shape[1]) for w in W],
        "g": [np.ones(WIDTH) for _ in range(k)],
        "be": [np.zeros(WIDTH) for _ in range(k)],
        "rm": [np.zeros(WIDTH) for _ in range(k)],
        "rv": [np.ones(WIDTH) for _ in range(k)],
    }


def norm_fwd(P, l, a, mode):
    """The l-th normalization layer. mode 'train': the batch's statistics + update the running mean; 'eval': the
    running mean."""
    if P["norm"] == "none":
        return a, None
    if mode == "eval":
        return P["g"][l] * ((a - P["rm"][l]) / np.sqrt(P["rv"][l] + EPS)) + P["be"][l], None
    mu, var = a.mean(0), a.var(0)
    inv = 1 / np.sqrt(var + EPS)
    if mode == "train":
        P["rm"][l] = (1 - MOM) * P["rm"][l] + MOM * mu
        P["rv"][l] = (1 - MOM) * P["rv"][l] + MOM * var * len(a) / (len(a) - 1)
    n = (a - mu) * inv
    return P["g"][l] * n + P["be"][l], (n, inv)


def norm_bwd(P, l, dy, c, grads):
    """The backward pass through BatchNorm (Lesson 10); without normalization, the gradient passes through unchanged."""
    if c is None:
        return dy
    n, inv = c
    grads["g"][l], grads["be"][l] = (dy * n).sum(0), dy.sum(0)
    dn = dy * P["g"][l]
    return inv / len(dn) * (len(dn) * dn - dn.sum(0) - n * (dn * n).sum(0))

In [ ]:
def forward(P, x, mode="train"):
    """Returns the cache for the backward pass, the last h, the logits z, and rms(h) after each hidden layer."""
    W, b, L = P["W"], P["b"], P["depth"]
    y, c = norm_fwd(P, 0, x @ W[0] + b[0], mode)  # layer 1 (784 → 64): a plain layer in both networks
    cache, h = [(x, y, c)], relu(y)
    rms = [np.sqrt((h**2).mean())]
    for l in range(1, L):
        if P["arch"] == "plain":  # h ← ReLU(N(hW + b))
            y, c = norm_fwd(P, l, h @ W[l] + b[l], mode)
            cache.append((h, y, c))
            h = relu(y)
        else:  # h ← h + ReLU(N(h))·W + b
            y, c = norm_fwd(P, l, h, mode)
            r = relu(y)
            cache.append((r, y, c))
            h = h + r @ W[l] + b[l]
        rms.append(np.sqrt((h**2).mean()))
    if P["arch"] == "res":  # N → ReLU once more before the output layer
        y, c = norm_fwd(P, L, h, mode)
        cache.append((None, y, c))
        h = relu(y)
    return cache, h, h @ W[-1] + b[-1], rms


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

In [ ]:
def backward(P, cache, h, z, y):
    """Backpropagation (Lesson 06) for both networks; returns the gradients of every parameter."""
    W, L = P["W"], P["depth"]
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)  # ∂L/∂z of softmax + cross-entropy, averaged over the batch
    k = len(P["g"])
    g = {"W": [None] * (L + 1), "b": [None] * (L + 1), "g": [None] * k, "be": [None] * k}
    g["W"][-1], g["b"][-1] = h.T @ dz, dz.sum(0)
    dh = dz @ W[-1].T
    if P["arch"] == "res":
        _, yl, c = cache[L]
        dh = norm_bwd(P, L, dh * (yl > 0), c, g)
    for l in range(L - 1, 0, -1):
        inp, yl, c = cache[l]
        if P["arch"] == "plain":  # the gradient has to go through the layer: multiply by Wᵀ, BN and ReLU
            da = norm_bwd(P, l, dh * (yl > 0), c, g)
            g["W"][l], g["b"][l] = inp.T @ da, da.sum(0)
            dh = da @ W[l].T
        else:  # the shortcut carries dh back unchanged; the branch F adds its own part
            g["W"][l], g["b"][l] = inp.T @ dh, dh.sum(0)
            dh = dh + norm_bwd(P, l, (dh @ W[l].T) * (yl > 0), c, g)
    x, yl, c = cache[0]
    da = norm_bwd(P, 0, dh * (yl > 0), c, g)
    g["W"][0], g["b"][0] = x.T @ da, da.sum(0)
    return g


def grad_profile(arch, norm, depth):
    """‖∂L/∂W_l‖ of every hidden layer and rms(h) at initialization, on the demo's 64-image batch (BN uses the batch's
    statistics)."""
    P = make_net(arch, norm, depth)
    cache, h, z, rms = forward(P, xb, "batch")
    gW = backward(P, cache, h, z, yb)["W"][:depth]
    return np.array([np.linalg.norm(gw) for gw in gW]), np.array(rms)

### 🤔 Predict first
A 30-layer plain network with BatchNorm, at initialization. Is the weight gradient at **layer 1** larger or smaller
than at **layer 30**, and by roughly how many times? Add skip connections (with the same weights): what does that
ratio become?

In [ ]:
prof = {(a, n): grad_profile(a, n, 30) for a in ("plain", "res") for n in ("bn", "none")}
for a, name in (("plain", "plain                "), ("res", "with skip connections")):
    gw, rms = prof[(a, "bn")]
    print(
        f"BatchNorm, {name}: ‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ = {sig(gw[0] / gw[-1]):>4}   "
        f"rms(h): layer 1 {sig(rms[0])} → layer 30 {sig(rms[-1])}"
    )
ratio_plain = prof[("plain", "bn")][0][0] / prof[("plain", "bn")][0][-1]
ratio_res = prof[("res", "bn")][0][0] / prof[("res", "bn")][0][-1]
print(
    f"plain network with BatchNorm: each step back through a layer makes the gradient grow by ×{num(ratio_plain ** (1 / 29), 3)} on average"
)
# resmlp.py computes exactly these two ratios on the same batch (899.343 and 38.5606); the demo shows them as 899, 38.6
assert abs(ratio_plain / 899.343 - 1) < 1e-5 and abs(ratio_res / 38.5606 - 1) < 1e-5, (
    "differs from the demo: check the data again"
)
assert (sig(ratio_plain), sig(ratio_res)) == ("899", "38.6")

The figure below plots the two networks with BatchNorm (log scale). On the left is
$\lVert\partial L/\partial W_l\rVert$ for each layer; the backward pass goes from right (layer 30) to left (layer 1).
On the right is the magnitude of the signal, $\mathrm{rms}(h)$, after each layer, in the forward pass. Blue, solid
line, round dots: with skip connections. Orange, dashed line, squares: the plain network.

In [ ]:
def two_lines(ax, a, b, every=1):
    layers = np.arange(1, len(a) + 1)
    ax.plot(layers, a, **RES, ms=4, lw=1.6, markevery=every, label="with skip connections: h ← h + F(h)")
    ax.plot(layers, b, **PLAIN, ms=4, lw=1.6, markevery=every, label="plain: h ← F(h)")
    ax.set_yscale("log")
    ax.grid(axis="y", alpha=0.3, which="major")


fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
two_lines(a1, prof[("res", "bn")][0], prof[("plain", "bn")][0])
a1.set(xlabel="layer l", ylabel="‖∂L/∂Wₗ‖", title="Gradient of each layer (← backward pass)")
two_lines(a2, prof[("res", "bn")][1], prof[("plain", "bn")][1])
a2.set(xlabel="layer l", ylabel="rms(h) after layer l", title="Signal after each layer (forward pass →)")
a1.legend(frameon=False, fontsize=9)
fig.suptitle(
    "30-layer MLP, BatchNorm, at initialization, on 64 images: the same weights, only the addition differs", fontsize=11
)
plt.tight_layout()
fmt_axes(fig)
plt.show()

At layer 1 of the plain network, the gradient is 899 times larger than at layer 30, exactly the number in Lesson 10
and in the demo. This is an **explosion**, not vanishing: BatchNorm keeps the forward signal at a constant size (the
orange line on the right is flat), but it is BatchNorm itself that makes the backward gradient grow at every layer.
Yang et al. (2019) prove this for plain networks with BatchNorm: no tuning of the initialization or of the activation
function removes it; with ReLU and a large batch, the squared norm of the gradient grows by a factor of
$\pi/(\pi - 1)$ per layer. The cell below compares that theoretical number with our measurement.

With the same weights and only the addition added, the ratio is down to 38.6. It still grows toward the first layers,
because each branch $F$ adds its own part of the gradient to $\partial L/\partial h$, but the part that goes through
the shortcut is not multiplied by any matrix.

In [ ]:
lam = math.pi / (math.pi - 1)
print(f"theory (very large batch): gradient norm ×√(π/(π−1)) = ×{num(math.sqrt(lam), 3)} per layer")
print(
    f"measured (batch of 64, 30 layers): ×{num(ratio_plain ** (1 / 29), 3)} per layer;  with skip connections: ×{num(ratio_res ** (1 / 29), 3)}"
)
rms_res = prof[("res", "bn")][1]
print(
    f"rms(h) with skip connections: layer 1 {num(rms_res[0], 3)} → layer 30 {num(rms_res[-1], 3)}; "
    f"if each branch adds a variance of 1: √({num(rms_res[0] ** 2, 2)} + 29) = {num(np.sqrt(rms_res[0] ** 2 + 29), 3)}"
)

Yang et al.'s theory is written for a very large batch and a very deep network, so we should only compare orders of
magnitude: ×1.21 against the ×1.26 we measured.
The signal of the network with skip connections, on the other hand, *grows* (the blue line on the right): each
branch receives an input that is already normalized (variance 1), after ReLU its mean square is ½, multiplied by the
He-initialized $W$ it becomes a variance of about 1, and then it is added to $h$. After 29 blocks,
$\mathrm{rms}(h) \approx \sqrt{\tfrac12 + 29}$, close to the measurement. The consequence: at the $l$-th block, the
branch $F$ only changes $h$ by about $1/\sqrt{l}$, so the deep blocks are almost the identity right from the start.
De & Smith (2020) argue that this is exactly why BatchNorm lets deep ResNets train.

## 6 · The addition alone is not enough

### 🤔 Predict first
Remove BatchNorm from both networks. At initialization, the gradients of the plain network and of the network with
skip connections are both fairly balanced between layer 1 and layer 30. Which network will learn within 1 epoch?

In [ ]:
for a, name in (("plain", "plain                "), ("res", "with skip connections")):
    gw, rms = prof[(a, "none")]
    print(
        f"no normalization, {name}: ‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ = {sig(gw[0] / gw[-1]):>3}   "
        f"rms(h): layer 1 {sig(rms[0])} → layer 30 {sig(rms[-1])}"
    )
rms_none = prof[("res", "none")][1]
print(
    f"with skip connections: rms(h) multiplied by ×{num((rms_none[-1] / rms_none[0]) ** (1 / 29), 2)} per layer (√2 = {num(math.sqrt(2), 2)})"
)
for d in mlp_runs["depths"]:  # the sweep's results: 4 values of η × 3 seeds for each depth
    rs = [r for r in mlp_runs["runs"] if (r["arch"], r["norm"], r["depth"]) == ("res", "none", d)]
    steps = [s["nanStep"] for r in rs for s in r["seeds"] if "nanStep" in s]
    eta, seeds = best_mlp("res", "none", d)
    best = f"; best η = {num_eta(eta)}: {mlp_text(seeds)}" if any("val" in s for s in seeds) else ""
    print(
        f"with skip connections, no normalization, {d:3d} layers: {len(steps)}/12 runs blew up (NaN), at step{'s' if max(steps) > min(steps) else ''} {min(steps)}{f'–{max(steps)}' if max(steps) > min(steps) else ''}{best}"
    )

At initialization, the gradients of both networks are balanced (ratios of 6.7 and 5.5), yet the network with skip
connections and no normalization **blows up** within its very first steps for every $\eta$ tried, from 20 layers up.
The reason lies in the forward pass's signal (the rms(h) lines printed above): each branch $F$ is He-initialized, so
the part it adds has a variance about the size of $h$'s variance; adding it **doubles** the variance at every block,
$\mathrm{rms}(h)$ is multiplied by about $\sqrt2$, and nothing pulls it back. After 29 blocks, the initial loss is
already huge, and so is the gradient, so one or two training steps are enough to push the network out of the range of
finite numbers. The plain network without normalization, on the other hand, at its best $\eta$, does not blow up, but
it learns poorly (77.0% in section 3).

So skip connections *on their own* are not enough; something has to keep the sum $h + F(h)$ from swelling. Real
ResNets put BatchNorm in every block. Another way is to initialize the branch $F$ close to 0, so that each block
starts out almost as the identity: Fixup (Zhang et al., 2019), SkipInit (De & Smith, 2020), and GPT-2, which
multiplies the weights of the layers on the residual branch by $1/\sqrt{N}$ at initialization. The “Try changing
things” section will try this.

## 7 · Training for real: 30 layers, 1 epoch

The gradient at initialization is only a sign. The cell below really trains the two 30-layer networks with
BatchNorm, as the demo does: SGD, batch 64, 1 epoch over 55,000 images in the order of `default_rng(1)`, seed 1. Each
network uses its best $\eta$ from the grid 0.01 · 0.03 · 0.1 · 0.3, chosen on the validation set in the sweep results.
Each network takes a few seconds.

In [ ]:
def accuracy(P, X, y):
    return float(
        np.mean([(forward(P, X[i : i + 2500], "eval")[2].argmax(1) == y[i : i + 2500]) for i in range(0, len(X), 2500)])
    )


def train(arch, norm, depth, eta, seed=1, branch_scale=1.0):
    """1 epoch of SGD; branch_scale multiplies the weights of the F branches at initialization (only used in the “Try
    changing things” section)."""
    np.seterr(all="ignore")  # a network that blows up overflows; we detect it from the loss below
    P = make_net(arch, norm, depth, seed)
    P["W"][1:depth] = [w * branch_scale for w in P["W"][1:depth]]
    order, losses = np.random.default_rng(seed).permutation(N_TRAIN), []
    for step, i in enumerate(range(0, N_TRAIN, BATCH), 1):
        idx = order[i : i + BATCH]
        cache, h, z, _ = forward(P, Xtr[idx], "train")
        zmax = z.max(1)  # a stable log-sum-exp, as in the demo's scripts
        losses.append((zmax + np.log(np.exp(z - zmax[:, None]).sum(1)) - z[np.arange(len(idx)), ytr[idx]]).mean())
        if not np.isfinite(losses[-1]):
            return None, step, losses
        g = backward(P, cache, h, z, ytr[idx])
        for k in ("W", "b", "g", "be"):
            for l, gk in enumerate(g[k]):
                if gk is not None:
                    P[k][l] -= eta * gk
    return P, step, losses

In [ ]:
nets, curves = {}, {}
for arch in ("plain", "res"):
    eta, seeds = best_mlp(arch, "bn", 30)
    t0 = time.perf_counter()
    nets[arch], _, curves[arch] = train(arch, "bn", 30, eta)
    acc = accuracy(nets[arch], Xv, yv)
    print(
        f"{'plain                ' if arch == 'plain' else 'with skip connections'} (η = {num_eta(eta)}): {pct(acc)} correct on the validation set "
        f"({num(time.perf_counter() - t0, 1)} s); sweep result, seed 1: {pct(seeds[0]['val'])}"
    )
    nets[arch]["acc"] = acc
assert abs(nets["res"]["acc"] - 0.9608) < 0.002, "differs from the demo: check the data again"
assert nets["plain"]["acc"] < nets["res"]["acc"] - 0.1

The figure below plots the training loss over that 1 epoch (a moving average over 20 steps, log scale): the solid
blue line is the network with skip connections, the dashed orange line the plain network.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for arch, st, label in (("res", RES, "with skip connections"), ("plain", PLAIN, "plain")):
    smooth = np.convolve(curves[arch], np.ones(20) / 20, mode="valid")
    ax.plot(np.arange(20, len(curves[arch]) + 1), smooth, color=st["color"], ls=st["ls"], lw=1.6, label=label)
ax.set_yscale("log")
ax.set(xlabel="step (1 epoch = 860 steps)", ylabel="training loss", title="30-layer MLP with BatchNorm, seed 1")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3, which="both")
fmt_axes(fig)
plt.show()

The network with skip connections learns right away: its number matches the sweep result that the demo stores. The
plain network lags far behind, even though the two networks have the same initial weights, the same data and the
same batch order.

For the plain network, your number may differ from the sweep result by a few percentage points, and the notebook
doesn't `assert` it. The reason is exactly what this lesson is about: the plain network's gradient explodes, so even
a rounding difference in the 16th digit (Colab runs on x86, the sweep ran on ARM) gets amplified into a different
training trajectory. The “Try changing things” section measures this.

## 8 · The sweep results: from 10 to 100 layers, and CNNs up to 56 layers

The full sweep (5 depths × 4 values of $\eta$ × 3 seeds for each network) takes a few minutes, so we read the results
that the demo has stored. Each depth takes the $\eta$ with the highest median validation accuracy.

In [ ]:
print("MLP, BatchNorm, 1 epoch — correct on the validation set, median [min–max] of 3 seeds:")
mlp_best = {}
for d in mlp_runs["depths"]:
    for arch in ("plain", "res"):
        mlp_best[(arch, d)] = best_mlp(arch, "bn", d)
    (ep, sp), (er, sr) = mlp_best[("plain", d)], mlp_best[("res", d)]
    print(
        f"  {d:3d} layers: plain {mlp_text(sp)} (η = {num_eta(ep)}) · with skip connections {mlp_text(sr)} (η = {num_eta(er)})"
    )

The figure below plots that table: each marker is one seed, and the line connects the medians. The dashed gray line at
10% is the level of random guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
D = mlp_runs["depths"]
for arch, st, label in (("res", RES, "with skip connections"), ("plain", PLAIN, "plain")):
    v = [[s["val"] for s in mlp_best[(arch, d)][1]] for d in D]
    for d, vs in zip(D, v):
        ax.scatter([d] * 3, 100 * np.array(vs), color=st["color"], marker=st["marker"], s=26, zorder=3)
    ax.plot(D, [100 * np.median(vs) for vs in v], **st, ms=5, mfc="white", lw=1.6, label=label)
ax.axhline(10, color=GREY, ls="--", lw=1)
ax.set_xscale("log")
ax.set(xlabel="number of hidden layers (log scale)", ylabel="correct on the validation set (%)", ylim=(0, 100))
ax.set_title("MLP with BatchNorm, 1 epoch, the best η for each depth")
ax.legend(frameon=False, loc="center left")
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
ax.xaxis.set_major_locator(FixedLocator(D))
plt.show()

With skip connections, the accuracy stays almost unchanged from 10 to 100 layers. Without them, the plain network
collapses from 30 layers on, and at 100 layers it is only around the level of random guessing. Note: skip connections
do not make the 100-layer network *better* than the 10-layer one here (96.4% versus 96.3%, within the variation
across seeds; MNIST and 1 epoch don't need that many layers); they make adding layers **no longer harmful**.

Now back to exactly Lesson 12's CNNs (section 3). `resnet.py` turns each pair of 3×3 convolutional layers into a
basic residual block of He et al. (2015): conv → BN → ReLU → conv → BN, add the shortcut, then ReLU. In the blocks
that shrink the image, the shortcut takes every second pixel and adds channels of zeros (option A). The two networks
have the same number of layers, the same initial weights, the same grid of $\eta$ and seeds, and **the same number of
parameters**, because the shortcut has no parameters at all.

In [ ]:
R_cnn = best_cnn(cnn_res)
for d in depths:
    p, r = P_cnn[d], R_cnn[d]
    print(
        f"{d:2d} layers ({num(p['params'][0], 0):>7} parameters): training error, plain {seed_text(p['train'])} · "
        f"with skip connections {seed_text(r['train'])}"
    )
for d in (20, 56):
    q = min((q for q in cnn_res["long"] if q["depth"] == d), key=lambda q: q["val"])
    print(
        f"10 epochs, seed 0, {d} layers with skip connections (η = {num_eta(q['eta'])}): training error {pct(q['train'])}, "
        f"test error {pct(q['test'])}"
    )
g = cnn_res["initGrads"]
print(
    f"gradient at initialization, first ÷ last convolutional layer, 56 layers: plain {sig(g['plain']['56'][0] / g['plain']['56'][-1])} · "
    f"with skip connections {sig(g['res']['56'][0] / g['res']['56'][-1])}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))
for grid, st, label in ((R_cnn, RES, "with skip connections"), (P_cnn, PLAIN, "plain (Lesson 12)")):
    for d in depths:
        ax.scatter([d] * 3, 100 * np.array(grid[d]["train"]), color=st["color"], marker=st["marker"], s=26, zorder=3)
    ax.plot(depths, [100 * np.median(grid[d]["train"]) for d in depths], **st, ms=5, mfc="white", lw=1.6, label=label)
ax.set(xticks=depths, xlabel="number of layers", ylabel="training error (%)", ylim=(0, None))
ax.set_title("Lesson 12's CNN on MNIST, 3 epochs, with skip connections added")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()

With skip connections, the training-error curve no longer rises with depth: at 56 layers, the 3-seed range of the
network with skip connections lies entirely below that of the plain network. Trained for 10 epochs (one seed), the
56-layer network has a *lower* training error than the 20-layer one, like what He et al. saw on ImageNet: ResNet-34
has a clearly lower training error than ResNet-18, and a top‑1 error of 25.03% versus 27.88% (Table 2). At 8 layers
the median of the network with skip connections is even slightly higher (1.81% versus 1.34%), but within the variation
across seeds: a shallow network has nothing to fix yet. The numbers are much smaller than in the paper because the
network is small and trained briefly; the shape of the curve is what to compare. The gradient at initialization of the
56-layer plain CNN is also off by nearly 1,000 times between the first and the last layer (946), like the plain MLP in
section 5; with skip connections, only 2.5 times.

## 9 · Deleting a block: a residual network as a collection of many paths

Veit et al. (2016) ran a simple experiment: take a trained network, **delete** one block (without retraining), then
measure the error. For a residual block, “delete” means keeping only the shortcut: $h \leftarrow h$. For a plain
network, deleting a $64 \to 64$ layer means connecting its input straight to the next layer.

### 🤔 Predict first
The two 30-layer networks just trained in section 7. Delete *one* layer in the middle (any of the 29 layers
64 → 64). Roughly how much accuracy does each network have left?

In [ ]:
def drop_layer(P, l):
    """A copy of the trained network without hidden layer l (1 … L−1): a residual block keeps only its shortcut; a
    plain layer is bypassed."""
    keep_W = [i for i in range(len(P["W"])) if i != l]
    keep_n = [i for i in range(len(P["g"])) if i != l]
    Q = {**P, "depth": P["depth"] - 1, "W": [P["W"][i] for i in keep_W], "b": [P["b"][i] for i in keep_W]}
    Q.update({k: [P[k][i] for i in keep_n] for k in ("g", "be", "rm", "rv")})
    return Q


Xl, yl = Xv[:2000], yv[:2000]  # 2,000 validation images to keep it fast, as the CNN experiment below uses 2,000 images
lesion = {arch: np.array([accuracy(drop_layer(nets[arch], l), Xl, yl) for l in range(1, 30)]) for arch in nets}
full_l = {arch: accuracy(nets[arch], Xl, yl) for arch in nets}
for arch, name in (("res", "with skip connections"), ("plain", "plain                ")):
    v = lesion[arch]
    print(
        f"{name}: full {pct(full_l[arch], 1)}; delete 1 layer → median {pct(np.median(v), 1)} "
        f"[{pct(v.min(), 1)}–{pct(v.max(), 1)}] over the 29 layers"
    )
print(f"network with skip connections: deleting layer {int(np.argmin(lesion['res'])) + 2} hurts the most")


def branch_share(P, X):
    """‖F(h)‖ / ‖h‖ of each block in the trained network with skip connections (inference mode)."""
    y, _ = norm_fwd(P, 0, X @ P["W"][0] + P["b"][0], "eval")
    h, out = relu(y), []
    for l in range(1, P["depth"]):
        y, _ = norm_fwd(P, l, h, "eval")
        f = relu(y) @ P["W"][l] + P["b"][l]
        out.append(np.linalg.norm(f) / np.linalg.norm(h))
        h = h + f
    return np.array(out)


share = branch_share(nets["res"], Xv[:2000])
print(
    f"‖F(h)‖ / ‖h‖: the block at layer 2 {num(share[0], 2)}; the blocks at layers 3–30 {num(share[1:].min(), 2)}–{num(share[1:].max(), 2)}"
)

The figure below plots the validation accuracy after deleting each layer (horizontal axis: the deleted layer), with
blue round dots for the network with skip connections and orange squares for the plain network; the two thin
horizontal lines are the accuracy of the full networks.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for arch, st, label in (
    ("res", RES, "with skip connections: delete a block"),
    ("plain", PLAIN, "plain: delete a layer"),
):
    ax.plot(np.arange(2, 31), 100 * lesion[arch], **st, ms=4.5, lw=1.2, label=label)
    ax.axhline(100 * full_l[arch], color=st["color"], ls=":", lw=1)
ax.set(xlabel="deleted layer", ylabel="correct on 2,000 validation images (%)", ylim=(0, 100), xlim=(1, 31))
ax.set_title("Trained 30-layer MLP: delete one layer, no retraining")
ax.legend(frameon=False, loc="center right")
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()

The plain network collapses to near random guessing even though it loses only one of its 30 layers: each layer
receives exactly the output of the layer before it, so removing one link means the whole chain after it receives
unfamiliar data. The network with skip connections barely changes, except for the first block (layer 2): there the
branch $F$ is even larger than the $h$ it is added to (the last line of the deletion cell, ‖F(h)‖ / ‖h‖ = 1.96),
while in the later blocks it is only a small correction.

The reason: expanding $h_3 = h_2 + F_3(h_2)$ with $h_2 = h_1 + F_2(h_1)$ shows that a network of $n$ blocks is a sum
of $2^n$ **paths**, each path either going through or skipping each block. Deleting one block removes only half of
the paths; the remaining paths still give a reasonable result. Veit et al. call this behaving like an **ensemble** of
shallower networks, and they measured that most of the gradient during training goes through the short paths: in a
110-layer ResNet, paths 10–34 layers long.

Veit's experiment used convolutional networks. `resnet.py` runs exactly that experiment on the two 56-layer CNNs of
section 8 ($\eta = 0.1$, 3 seeds), measuring the error on 2,000 test images: deleting blocks one at a time, then
deleting $k$ random blocks at once (20 draws for each $k$). In the plain network, the size-preserving
Conv → BN → ReLU layers are deleted one at a time (52 such layers).

In [ ]:
def med(a):
    """The median as in the demo: with an even number of elements, take the upper of the two middle ones."""
    return sorted(a)[len(a) // 2]


LS = cnn_res["lesion"]
print(
    f"{LS['depth']} layers, η = {num_eta(LS['eta'])}; error measured on {num(LS['nTest'], 0)} test images, {LS['draws']} draws for each k"
)
full = {a: [r["full"] for r in LS[a]] for a in ("res", "plain")}
single = {a: [med([r["single"][u] for r in LS[a]]) for u in range(LS[a][0]["units"])] for a in ("res", "plain")}
for a, name, unit in (("res", "CNN with skip connections", "block"), ("plain", "plain CNN", "layer")):
    print(
        f"{name}: {LS[a][0]['units']} deletable {unit}s; full test error {seed_text(full[a], 1)}, delete 1 {unit} → "
        f"median {pct(med(single[a]), 1)} (per {unit}: {pct(min(single[a]), 1)}–{pct(max(single[a]), 1)})"
    )
heavy = [(u + 1, v) for u, v in enumerate(single["res"]) if v > 3 * med(full["res"])]
print(
    "blocks whose deletion raises the error more than 3-fold:", ", ".join(f"block {u} ({pct(v, 1)})" for u, v in heavy)
)
per_stage = LS["res"][0]["units"] // 3
print(
    "downsampling blocks:", ", ".join(f"block {u + 1} ({pct(single['res'][u], 1)})" for u in (per_stage, 2 * per_stage))
)


def pooled(a):
    """For each k: every draw of all 3 seeds pooled together (k = 0 is the full network)."""
    return [full[a]] + [list(itertools.chain(*(r["random"][i] for r in LS[a]))) for i in range(len(LS["ks"]))]


series = {a: [med(v) for v in pooled(a)] for a in full}
print(
    "delete k random blocks, median error:",
    " · ".join(f"k = {k}: {pct(v, 1)}" for k, v in zip([0, *LS["ks"]], series["res"])),
)

The figure below plots the test error of the two 56-layer CNNs against the number of blocks (or layers) deleted at
random at once: the line is the median of 60 draws (3 seeds × 20), and the shaded band holds the middle half of the
draws (from the lower quartile to the upper quartile). Blue round dots: with skip connections; orange squares: the
plain network.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ks = [0, *LS["ks"]]
for a, st, label in (("res", RES, "residual: delete k blocks"), ("plain", PLAIN, "plain: delete k layers")):
    draws = [sorted(v) for v in pooled(a)]
    q = np.array([[d[round(f * (len(d) - 1))] for f in (0.25, 0.75)] for d in draws])
    ax.fill_between(ks, 100 * q[:, 0], 100 * q[:, 1], color=st["color"], alpha=0.15, lw=0)
    ax.plot(ks, 100 * np.array(series[a]), **st, ms=4.5, lw=1.6, label=label)
ax.axhline(90, color=GREY, ls="--", lw=1)
ax.text(0.1, 92, "random guessing", color=GREY, fontsize=9, ha="left")
ax.set(xticks=ks, xlabel="number of blocks / layers deleted", ylabel="test error (%)", ylim=(0, 100))
ax.set_title("Trained 56-layer CNN, 2,000 test images: random deletion, no retraining")
ax.legend(frameon=False, loc="center right")
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()

The CNNs paint the same picture: deleting one layer of the plain network breaks it completely, while deleting one
block of the residual network usually raises the error only slightly. The error rises **gradually** as more blocks
are deleted, rather than collapsing all at once, just as Veit et al. describe. The first three blocks of
stage 1 are the exception, like the first block of the MLP above. Veit found that deleting a downsampling block has a
moderate effect, more noticeable than for the other blocks; here those two blocks (blocks 10 and 19) give 3.4% and
4.9%, slightly above the median. Deleting 10 blocks raises the error to 84.4%, much higher than in the paper, because
here 10 blocks are more than a third of the network (the paper deletes 10 of 54 blocks).

Veit et al. draw a memorable conclusion: ResNet does not keep the gradient intact through the *whole* depth; it adds
short paths that carry the gradient through the network, and training happens mainly along those paths.

## 10 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again; each
training run takes a few seconds.

### Skip connections without BatchNorm, if the branch starts small
Section 6 found that the network with skip connections and no normalization blows up right at step 3. The idea behind
Fixup and SkipInit: multiply the weights of each branch $F$ by a small number at initialization, so that $h + F(h)$
starts close to $h$. The cell below tries three factors on the 30-layer network without normalization,
$\eta = 0.03$ (including $1/\sqrt{29}$, about the factor GPT-2 uses).

In [ ]:
pred = {}
for scale in (1.0, 1 / np.sqrt(29), 0.0):
    P, step, losses = train("res", "none", 30, 0.03, branch_scale=scale)
    if P is None:
        print(f"branch F initialized ×{num(scale, 3)}: blew up (NaN) at step {step}")
        assert scale == 1.0 and step <= 4, (
            "differs from the demo: the network without normalization must blow up within the first few steps"
        )
        continue
    pred[scale] = forward(P, Xv, "eval")[2].argmax(1)
    print(f"branch F initialized ×{num(scale, 3)}: {pct((pred[scale] == yv).mean())} correct on the validation set")
a, b = pred.values()
print(f"the two networks that learned give the same answer for {pct((a == b).mean())} of the validation images")

The two accuracies being equal is only a coincidence: the two networks give different answers on a few percent of
the images. With the branch multiplied by 0 at initialization, the network starts out as just layer 1 plus the output
layer; the blocks grow during training. Both ways train 30 layers with skip connections without needing BatchNorm.
The real condition is that the sum $h + F(h)$ must not swell with depth; BatchNorm is one way, initializing the branch
small is another.

### The plain network is sensitive to a tiny difference
Section 7 said that the plain network's numbers may differ on another machine. The cell below measures that: it runs
each of the two networks for 100 steps, once with the original weights and once with $W_1$ multiplied by
$(1 + 10^{-12}\varepsilon)$, then measures the relative distance between the two sets of weights.

In [ ]:
def drift(arch, eta, steps=100, rel=1e-12):
    runs = []
    for r in (0.0, rel):
        P = make_net(arch, "bn", 30)
        P["W"][0] = P["W"][0] * (1 + r * np.random.default_rng(99).standard_normal(P["W"][0].shape))
        order = np.random.default_rng(1).permutation(N_TRAIN)
        for i in range(0, steps * BATCH, BATCH):
            idx = order[i : i + BATCH]
            cache, h, z, _ = forward(P, Xtr[idx], "train")
            g = backward(P, cache, h, z, ytr[idx])
            for k in ("W", "b", "g", "be"):
                for l, gk in enumerate(g[k]):
                    if gk is not None:
                        P[k][l] -= eta * gk
        runs.append(np.concatenate([w.ravel() for w in P["W"]]))
    return np.linalg.norm(runs[1] - runs[0]) / np.linalg.norm(runs[0])


for arch, name in (("plain", "plain                "), ("res", "with skip connections")):
    print(
        f"{name}: after 100 steps, the two sets of weights are {sig(drift(arch, best_mlp(arch, 'bn', 30)[0]))} apart (relative)"
    )

After 100 steps, the $10^{-12}$ difference in the plain network has grown to about one fifth of the size of the
weights: the two runs are two completely different networks. In the network with skip connections, the difference
stays at its initial size. This is another face of the same problem: a deep plain network is a noise amplifier, to
the point that its result depends on how the machine rounds numbers. Balduzzi et al. (2017) describe a phenomenon
close to this one in the gradient: in a deep plain network, the gradients of nearby inputs become almost uncorrelated
(“shattered gradients”), while skip connections keep them correlated.

### Other depths
Change `depth` to see how the gradient ratio at initialization changes with depth (the demo lets you choose 10, 20,
30, 50 or 100 layers).

In [ ]:
for depth in (10, 20, 50, 100):
    rp, rr = (grad_profile(a, "bn", depth)[0] for a in ("plain", "res"))
    print(
        f"{depth:3d} layers, BatchNorm: ‖∂L/∂W₁‖ / ‖∂L/∂W_L‖ plain {sig(rp[0] / rp[-1]):>9} · with skip connections {sig(rr[0] / rr[-1])}"
    )

In the plain network, the ratio grows exponentially with depth (each layer multiplies by an almost fixed factor); with
skip connections it only grows slowly. Try also: change `seed` in section 7's `train(...)`, or change `eta`, then
compare with the 3-seed range in section 8.

## 11 · Limits and next steps

- **Skip connections fix optimization, not overfitting.** On CIFAR-10, He et al. trained a 1,202-layer network
  (19.4 million parameters), but its test error of 7.93% is higher than that of the 110-layer network (6.43%); the
  authors attribute this to overfitting.
- **Not the only explanation.** Why skip connections help is still debated: a path that lets the gradient go straight
  through, blocks that easily learn the identity, an ensemble of short paths (Veit), gradients that don't shatter
  (Balduzzi), a loss landscape that is easier to optimize (Li et al., 2018). These views complement each other.
- **Real architectures.** ResNet-18/34 use section 8's basic block (ReLU after the addition); ResNet-50/101/152 use
  the 1×1 → 3×3 → 1×1 bottleneck block to reduce the cost. Highway Networks (Srivastava et al., 2015) had shortcuts
  before that, but with learned gates.

The last cell counts the parameters of 5 ResNets for ImageNet from their structure (color images, 1,000 classes), to
show that “152 layers” does not mean 8 times as large as “18 layers”.

In [ ]:
def resnet_params(blocks, bottleneck):
    """The parameters of a ResNet for ImageNet (He et al. 2015, Table 1): a 7×7 conv, 4 stages, a 1,000-class dense
    layer. The convolutions have no bias (BN follows); BN has γ, β; shortcuts that change size use a 1×1 convolution
    (option B)."""
    conv = lambda cin, cout, k: cin * cout * k * k + 2 * cout  # weights + γ, β of the BN that follows
    p, cin, mult = conv(3, 64, 7), 64, 4 if bottleneck else 1
    for stage, (n, w) in enumerate(zip(blocks, (64, 128, 256, 512))):
        for i in range(n):
            cout = w * mult
            if bottleneck:  # bottleneck block 1×1 → 3×3 → 1×1
                p += conv(cin, w, 1) + conv(w, w, 3) + conv(w, cout, 1)
            else:  # basic block 3×3 → 3×3
                p += conv(cin, w, 3) + conv(w, w, 3)
            if cin != cout or (stage > 0 and i == 0):
                p += conv(cin, cout, 1)
            cin = cout
    return p + cin * 1000 + 1000


CONFIGS = {18: ([2, 2, 2, 2], False), 34: ([3, 4, 6, 3], False), 50: ([3, 4, 6, 3], True),
           101: ([3, 4, 23, 3], True), 152: ([3, 8, 36, 3], True)}  # fmt: skip
counts = {d: resnet_params(*c) for d, c in CONFIGS.items()}
for d, n in counts.items():
    print(f"ResNet-{d:<3}: {num(n, 0):>10} parameters ({num(n / counts[18], 1)} times ResNet-18)")
# the parameter counts that torchvision publishes for its models
assert list(counts.values()) == [11689512, 21797672, 25557032, 44549160, 60192808]


def cifar_params(depth):
    """A ResNet for CIFAR-10 (section 4.2 of the paper): 3 stages of 16/32/64 channels, basic blocks, option-A
    shortcuts."""
    conv = lambda cin, cout: cin * cout * 9 + 2 * cout
    p, cin = conv(3, 16), 16
    for w in (16, 32, 64):
        for _ in range((depth - 2) // 6):
            p, cin = p + conv(cin, w) + conv(w, w), w
    return p + 64 * 10 + 10


for d in (110, 1202):
    print(f"{num(d, 0)}-layer ResNet for CIFAR-10: {num(cifar_params(d) / 1e6, 1)} million parameters")

ResNet-152 is more than 8 times as deep as ResNet-18 but has only about 5 times as many parameters, thanks to the
bottleneck block. That is also why ResNet-50 has only slightly more parameters than ResNet-34, even though it is 16
layers deeper.

Skip connections don't stop at computer vision. In today's Transformers (Lesson 17), every attention block and every
MLP block is written $h \leftarrow h + F(\mathrm{Norm}(h))$, exactly section 5's pre-activation form: a “residual
stream” runs from the embeddings to the output layer, and each block only adds its own correction. Lesson 14 begins
that stretch of the course, with the Transformer's input: tokens and embeddings.

## Summary
- **The degradation problem:** the 56-layer plain CNN has a *training* error of 9.41% versus 0.99% for the 20-layer
  network. It is a failure of optimization, not overfitting, and BatchNorm doesn't rescue it.
- **Residual block** $y = x + F(x)$: with $F = 0$ it is the identity, and its Jacobian is $I + \partial F/\partial x$.
- **At initialization:** in the 30-layer plain MLP with BatchNorm, the gradient at layer 1 is 899 times the gradient
  at layer 30 (an explosion, not vanishing). Same weights, plus the addition: 38.6 times.
- **The addition alone is not enough:** without normalization, the variance of $h$ doubles at every block and the
  network blows up at step 3. BatchNorm, or initializing the branch small (Fixup, SkipInit), keeps the sum from
  swelling.
- **Training for real:** 30-layer MLP, 1 epoch: 96.08% with skip connections versus 65.96% plain (seed 1). With skip
  connections, the accuracy is almost unchanged from 10 to 100 layers. The 56-layer CNN with skip connections has a
  training error of 0.99% [0.78%–1.34%], on par with the 20-layer plain network (0.99% [0.87%–1.10%]), instead of
  9.41%.
- **Deleting a block** of a trained residual network usually raises the error only slightly; deleting a layer of a
  plain network makes it collapse to near random guessing. A residual network is like an ensemble of many short paths
  (Veit et al., 2016).

## Review questions
1. The 56-layer network has a higher training error than the 20-layer one. Why is this not overfitting?
2. Write the derivative $\partial y/\partial x$ of $y = x + \mathrm{ReLU}(x)W + b$. Which part guarantees that the
   gradient is not wiped out when it passes through the block?
3. A 30-layer plain network with BatchNorm: is the gradient at layer 1 larger or smaller than at layer 30? What does
   that say about the claim “ResNet fixes vanishing gradients”?
4. Why does the network with skip connections and no normalization blow up, even though its gradient at
   initialization is even more balanced than the plain network's? Name two fixes.
5. Deleting one block of a trained ResNet raises the error only slightly; deleting one layer of a plain network makes
   it collapse. Explain with the number of paths.
6. How many parameters does the shortcut have? Why does this matter when comparing the plain network with the network
   with skip connections in section 8?

<details><summary>Hints and answers</summary>

1. Overfitting means a low training error but a high test error. Here it is the training error itself that is high,
   even though the 56-layer network could do at least as well as the 20-layer one (by having the extra layers compute
   the identity): the optimization fails.
2. $\partial y_j/\partial x_i = \delta_{ij} + \mathbb 1[x_i > 0]\,W_{ij}$, that is, $I + \partial F/\partial x$. The
   shortcut's $I$ part carries $\partial L/\partial y$ back intact, without multiplying it by any matrix.
3. Larger, 899 times: BatchNorm makes the plain network's gradient explode with depth (Yang et al., 2019). He et al.
   also checked that the gradient of a plain network with BatchNorm does not vanish. The problem of deep plain
   networks is not just a small gradient.
4. Each He-initialized branch adds to $h$ a part whose variance is about the size of $h$'s variance, so the variance
   doubles at every block ($\mathrm{rms}(h)$ ×1.42 per layer); the initial loss and gradient are huge, so a few
   training steps are enough to overflow.
   Fixes: BatchNorm in every block, or initializing the branch $F$ close to 0 (Fixup, SkipInit, GPT-2's $1/\sqrt{N}$
   factor).
5. A network of $n$ blocks is a sum of $2^n$ paths; deleting one block leaves $2^{n-1}$ paths, which still give a
   reasonable result. A plain network has only one path; remove one link and it breaks.
6. None at all (option A). The two networks have exactly the same number of parameters and the same initial weights,
   so the difference can only come from the addition.
</details>

## Further reading
- K. He, X. Zhang, S. Ren, J. Sun (2015), [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385):
  the degradation problem, the residual block, options A/B, CIFAR-10 up to 1,202 layers.
- K. He et al. (2016), [Identity Mappings in Deep Residual Networks](https://arxiv.org/abs/1603.05027): the
  pre-activation block, the shortcut as a pure identity.
- A. Veit, M. Wilber, S. Belongie (2016), [Residual Networks Behave Like Ensembles of Relatively Shallow Networks](https://arxiv.org/abs/1605.06431):
  the block-deletion experiment, the short paths that carry the gradient.
- G. Yang et al. (2019), [A Mean Field Theory of Batch Normalization](https://arxiv.org/abs/1902.08129): BatchNorm
  makes the gradient of deep plain networks explode.
- S. De, S. Smith (2020), [Batch Normalization Biases Residual Blocks Towards the Identity Function in Deep Networks](https://arxiv.org/abs/2002.10444)
  and H. Zhang, Y. Dauphin, T. Ma (2019), [Fixup Initialization](https://arxiv.org/abs/1901.09321): residual networks
  without normalization.
- D. Balduzzi et al. (2017), [The Shattered Gradients Problem](https://arxiv.org/abs/1702.08591);
  H. Li et al. (2018), [Visualizing the Loss Landscape of Neural Nets](https://arxiv.org/abs/1712.09913).
- R. Srivastava, K. Greff, J. Schmidhuber (2015), [Highway Networks](https://arxiv.org/abs/1505.00387): gated
  shortcuts.
- d2l.ai, [Residual Networks (ResNet)](https://d2l.ai/chapter_convolutional-modern/resnet.html): the same idea through
  nested function classes, with ResNet-18 code.

**Next lesson:** [Lesson 14 · Tokens & embeddings](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-embedding/):
turning text into a sequence of vectors, the input of the Transformer.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")